# Brick 1 7B Scale-Up -- Colab Pro+ (A100 40GB)

Fresh LoRA (r=32) on Qwen2.5-7B fp16 + same drill data + same 6 gates.

**No warm-start:** Brick 0 adapter is on 0.5B base -- dim mismatch with 7B. This is a scale-up, not continual.

**Inputs (HF):** `issdandavis/scbe-drill-langues-full` (2630 rows). 60/40 replay mix built in-notebook.
**Output (HF):** `issdandavis/tongue-table-lora-brick1-7b-v1`.

**VRAM budget (A100 40GB):** 7B fp16 weights ~14GB + LoRA r=32 + optimizer + micro_batch=1 block_size=256 activations ~= 18-22 GiB used. Fits.

## 1. Environment

In [ ]:
import os, subprocess, sys
REPO_URL = 'https://github.com/issdandavis/SCBE-AETHERMOORE.git'
BRANCH = 'neurogolf/ant-colony-solvers'
REPO = '/content/SCBE-AETHERMOORE'
if not os.path.isdir(REPO):
    subprocess.check_call(['git','clone','--depth','1','--branch',BRANCH,REPO_URL,REPO])
else:
    subprocess.check_call(['git','-C',REPO,'fetch','--depth','1','origin',BRANCH])
    subprocess.check_call(['git','-C',REPO,'reset','--hard',f'origin/{BRANCH}'])
os.chdir(REPO); sys.path.insert(0, REPO)
subprocess.check_call(['git','log','-1','--oneline'])

In [ ]:
!pip install -q 'transformers>=4.46' 'peft>=0.13' 'accelerate>=1.0' 'datasets>=3.0' 'huggingface_hub>=0.26' 'safetensors' 'sentencepiece'

In [ ]:
import torch
assert torch.cuda.is_available(), 'Need GPU -- Runtime -> Change runtime type -> A100'
p = torch.cuda.get_device_properties(0)
vram_gib = p.total_memory / 1024**3
print(f'GPU: {p.name}  VRAM: {vram_gib:.1f} GiB')
if vram_gib < 35:
    print('WARN: <35 GiB -- 7B fp16 will OOM. Use A100 40GB, or switch to lora_r=8 + grad_accum=32 on L4.')

## 2. HF auth

In [ ]:
from huggingface_hub import login, whoami
import os
if 'HF_TOKEN' not in os.environ:
    try:
        from google.colab import userdata
        for key in ('huggingfacewrite','HF_TOKEN','huggingfaceread'):
            try:
                tok = userdata.get(key)
                if tok:
                    os.environ['HF_TOKEN'] = tok
                    print(f'secret: {key}')
                    break
            except Exception:
                continue
    except ImportError:
        pass
if 'HF_TOKEN' not in os.environ:
    from getpass import getpass
    os.environ['HF_TOKEN'] = getpass('HF token (write scope): ')
login(token=os.environ['HF_TOKEN'], add_to_git_credential=False)
print('auth:', whoami()['name'])

## 3. Pull drill data

In [ ]:
from huggingface_hub import hf_hub_download
from pathlib import Path
drill = Path('data/tongue_drill/drill_langues_full.jsonl')
drill.parent.mkdir(parents=True, exist_ok=True)
hf_hub_download(
    repo_id='issdandavis/scbe-drill-langues-full', repo_type='dataset',
    filename='drill_langues_full.jsonl', local_dir=str(drill.parent),
)
print('drill rows:', sum(1 for _ in open(drill, encoding='utf-8')))

## 4. Build 60/40 mix (same as 0.5B run)

In [ ]:
!python scripts/build_brick1_boost.py
import sys; sys.path.insert(0, '.')
from scripts.train_brick1_continual import build_mix
stats = build_mix()
print('mix stats:', stats)

## 5. Fresh LoRA fit on Qwen2.5-7B -- fp16, r=32, 750 steps

Direct trainer call (not `train_brick1_continual.py`) because:
- No resume adapter (different base)
- Smaller `batch_size=1` + higher `grad_accum=16` to fit 40GB VRAM
- LoRA r bumped 16 -> 32 (more capacity to absorb 7B)

In [ ]:
import json, subprocess
cmd = [
    'python','scripts/train/lora_tongue_table.py',
    '--base_model','Qwen/Qwen2.5-7B',
    '--data','data/tongue_drill/brick1_mix.jsonl',
    '--output','artifacts/tongue-table-lora-brick1-7b-v1',
    '--max_steps','750',
    '--eval_every','25',
    '--lr','1e-4',
    '--lr_scheduler_type','cosine',
    '--warmup_steps','25',
    '--early_stop_score','0.90',
    '--lora_r','32','--lora_alpha','64',
    '--batch_size','1','--grad_accum','16',
    '--block_size','256',
    '--map_weights', json.dumps({'transport_atomic':1.0,'cartography_state':2.0,'cross_braid_code':1.5}),
    '--default_map_weight','1.0',
]
print(' '.join(cmd))
rc = subprocess.call(cmd)
print('trainer rc:', rc)
assert rc == 0, f'trainer failed with rc={rc}'

## 6. Gate eval (absolute thresholds per plan §4)

In [ ]:
import subprocess
proc = subprocess.run([
    'python','scripts/eval_brick1_gates.py',
    '--adapter','artifacts/tongue-table-lora-brick1-7b-v1/lora_final',
    '--base_model','Qwen/Qwen2.5-7B',
    '--output','artifacts/tongue-table-lora-brick1-7b-v1/brick1_gate_report.json',
])
print('exit:', proc.returncode)
print({0:'GREEN',1:'AMBER',2:'RED',3:'SETUP ERROR'}.get(proc.returncode,'?'))

In [ ]:
import json
r = json.load(open('artifacts/tongue-table-lora-brick1-7b-v1/brick1_gate_report.json'))
print('overall:', r['overall_status'])
for g in r['gates']:
    v = g.get('brick1_value')
    vs = f'{v:.4f}' if isinstance(v, float) else 'N/A'
    print(f"  [{g['status']:7s}] {g['gate']:22s} 7b={vs}  {g['reason']}")

## 7. Push to HF

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
target = 'issdandavis/tongue-table-lora-brick1-7b-v1'
api.create_repo(repo_id=target, repo_type='model', exist_ok=True, private=False)
api.upload_folder(
    folder_path='artifacts/tongue-table-lora-brick1-7b-v1',
    repo_id=target, repo_type='model',
    commit_message='Brick 1 7B scale-up LoRA + gate report',
)
print('->', 'https://huggingface.co/' + target)